# Программирование на языке Python
# Неделя 8. Краулеры

## 1. Как клиент общается с сервером

**Клиент** — программа, которая запрашивает ресурс. Это может быть браузер,
мобильное приложение или наш Python-скрипт.

**Сервер** — программа, которая принимает запросы и отправляет ответы.
Словом «сервер» также называют компьютер, на котором работает такая программа.
Клиент и сервер могут работать даже на одном компьютере.

Браузер не просто получает HTML: он обычно делает дополнительные запросы за стилями,
картинками и скриптами, выполняет JavaScript и рисует страницу.


### 1.1. Как адрес помогает обратиться к серверу

Чтобы получить страницу, клиенту нужно определить, куда отправить запрос,
как установить связь с сервером и какой ресурс запросить. Эти сведения
содержатся в **URL** (Uniform Resource Locator) — адресе ресурса.
Разберём адрес по частям, а затем свяжем их в последовательность действий.

```text
https://example.org:443/catalog/books?page=2&lang=ru#reviews
└─┬─┘  └────┬────┘ └┬┘└─────┬─────┘└──────┬──────┘└──┬──┘
 схема     домен   порт      путь       параметры    фрагмент
```

#### IP-адрес: куда направлять данные в сети

Интернет — сеть связанных между собой сетей. Данные передаются в виде
**пакетов** — отдельных порций с полезным содержимым и служебной информацией.

**IP** (Internet Protocol) — протокол, определяющий адресацию и передачу таких
пакетов между сетями. **IP-адрес** — числовой адрес сетевого интерфейса, то есть
подключения устройства к сети. Это не постоянный «номер компьютера»: адрес может
меняться, а у одного устройства может быть несколько интерфейсов и адресов.

Сейчас используются две версии IP:

| Версия | Размер адреса | Пример записи | Как читать запись |
|---|---|---|---|
| **IPv4** (версия 4) | 32 бита | `192.0.2.1` | Четыре десятичных числа от 0 до 255, разделённые точками |
| **IPv6** (версия 6) | 128 бит | `2001:db8::1` | Группы шестнадцатеричных цифр, разделённые двоеточиями; `::` сокращает последовательность нулевых групп |

IPv4 допускает около 4,3 миллиарда комбинаций,
IPv6 — существенно больше. Ограниченность адресного пространства IPv4 стала
одной из причин разработки IPv6. 

#### Домен и DNS: как имя связано с IP-адресом

В адресе сайта обычно указано **доменное имя**, например `example.org`,
а не IP-адрес. **DNS** (Domain Name System) — распределённая система, которая,
в частности, позволяет получить IP-адреса по доменному имени. Клиент обращается
к DNS, получает адрес и использует его для подключения. 

Соответствие не обязательно взаимно однозначное: одно имя может иметь несколько
IP-адресов, а по одному IP-адресу могут обслуживаться разные сайты. 

#### Порт: к какой службе обратиться по этому адресу

На одном компьютере могут одновременно работать веб-сервер, сервер базы данных
и другие сетевые программы. Одного IP-адреса недостаточно, чтобы различать
обращения к этим службам. Для этого используются **порты** — числовые обозначения
в транспортных протоколах, которые обеспечивают передачу данных между
программами. Сочетание IP-адреса, транспортного протокола и номера порта позволяет
операционной системе направить входящие данные соответствующей сетевой службе.

Номер порта находится в диапазоне от 0 до 65535.
Порт — не физический разъём и не отдельная программа. Когда говорят, что сервер
**слушает порт**, это означает, что программа настроена принимать на нём обращения.

В URL порт записывается после имени через двоеточие: `https://example.org:443/`.
Если он не указан, клиент выбирает стандартный для схемы адреса порт:
**80 для HTTP**, **443 для HTTPS**. Поэтому `https://example.org/` и
`https://example.org:443/` указывают на один и тот же адрес. Для нестандартного
порта номер нужен явно: `http://localhost:8000/`. Имя `localhost` обозначает
этот же компьютер, а не внешний сервер.

#### HTTP и HTTPS: по каким правилам передаются запросы и ответы

**Протокол** — согласованный набор правил: как устроены сообщения и как
участники обмена должны их обрабатывать. **HTTP** (Hypertext Transfer Protocol)
задаёт правила запросов к веб-ресурсам и ответов на них. Несмотря на название,
он позволяет передавать не только HTML, но и JSON, изображения и другие данные.

Обычный HTTP сам по себе не шифрует сообщения: участник сети, имеющий доступ
к передаваемому трафику, может прочитать или изменить их. **HTTPS** — защищённый
вариант HTTP: для защиты используется **TLS** (Transport Layer Security).
TLS обеспечивает шифрование, проверку целостности передаваемых данных и,
при корректной проверке сертификата, подтверждение подлинности сервера.

**Сертификат сервера** связывает доменное имя с открытым криптографическим ключом.
Клиент проверяет, в частности, соответствие имени, срок действия сертификата
и цепочку доверия к удостоверяющему центру. Это часть проверки того, что
защищённое соединение устанавливается с сервером нужного домена.
Браузер и `requests` обычно выполняют эти проверки автоматически.
Отключение проверки через `verify=False` лишает соединение этой защиты
от подмены сервера.

HTTPS защищает **передачу**, но не гарантирует добросовестность владельца сайта,
достоверность его материалов или разрешение на сбор данных.
Шифрование также не скрывает все сведения о соединении, например IP-адрес получателя.

#### Путь, параметры и фрагмент: что именно запрашиваем

- **Путь** в примере — `/catalog/books`. Сервер использует его для выбора
  ресурса или обработчика запроса. Это может быть готовый файл, но может быть
  и страница, которую программа создаёт по данным из базы. Поэтому `/catalog/books`
  не означает, что на диске обязательно есть папка `catalog` и файл `books`.
- **Query-параметры** — часть адреса после `?`: `page=2&lang=ru`. Обычно это пары
  `имя=значение`, разделённые `&`. Здесь клиент передаёт значения `2` для `page`
  и `ru` для `lang`. Их смысл определяет сайт: можно предположить номер страницы
  и язык, но сервер должен поддерживать именно такие параметры. 
- **Фрагмент** — часть после `#`: `reviews`. Браузер может использовать его,
  чтобы прокрутить уже загруженную страницу до элемента с `id="reviews"`. Фрагмент не отправляется серверу как часть адреса в HTTP-запросе: `/catalog/books#reviews` и
  `/catalog/books#description` запрашивают один и тот же ресурс.

#### Что происходит при открытии адреса

Для нового HTTPS-соединения основная последовательность такова:

1. Клиент разбирает URL: выделяет схему, домен, порт, путь и параметры.
2. Определяет IP-адрес по доменному имени с помощью DNS или использует кэш.
3. Устанавливает соединение с сервером по выбранному адресу и порту.
4. Согласует защиту TLS и проверяет сертификат сервера.
5. Отправляет HTTP-запрос с путём и параметрами, например
   `GET /catalog/books?page=2&lang=ru`, а также заголовками, включая имя сайта.
6. Получает HTTP-ответ: статус, заголовки и тело, например HTML.

Существующее соединение может использоваться повторно, поэтому DNS-поиск
и установка нового соединения происходят не для каждого запроса.


### 1.2. HTTP: запрос и ответ

**HTTP** — протокол, то есть правила обмена сообщениями между клиентом и сервером.

HTTP различает два вида сообщений: **запрос** отправляет клиент,
**ответ** возвращает сервер. Рассмотрим их составляющие:

- **Метод запроса** указывает действие, например получить ресурс (`GET`).
- **Адрес ресурса в запросе** указывает, к чему относится действие:
  в нашем примере это путь `/catalog/books` с параметрами `page=2&lang=ru`.
- **Тело сообщения** — собственно передаваемые данные. В ответе это может быть
  HTML страницы, а в запросе — данные заполненной формы. 
- **Код состояния ответа** — трёхзначное число, сообщающее результат обработки
  запроса: например, `200` означает успех, а `404` — что ресурс не найден.

| Метод | Назначение |
|---|---|
| `GET` | Получить ресурс: страницу, файл, данные |
| `POST` | Отправить данные, например заполненную форму |


**Заголовки** — технические сведения. Например, `User-Agent` описывает клиента,
`Content-Type` — формат тела, а `Retry-After` может сообщать, когда повторить запрос.


| Код | Что означает | 
|---|---|---|
| `200` | Запрос успешно обработан |
| `301`, `302` | Перенаправление | 
| `401` | Требуется аутентификация | 
| `403` | Доступ запрещён | 
| `404` | Ресурс не найден | 
| `429` | Слишком много запросов | 
| `500`, `503` | Ошибка или недоступность сервера | 


## 2. Первые запросы с requests

Работаем с [Quotes to Scrape](https://quotes.toscrape.com/) — учебным сайтом
для тренировки сбора данных. Соберём небольшой корпус: текст цитаты, автора,
теги и адрес источника. 

In [ ]:
# !pip install requests

In [33]:
import time
from urllib.parse import urljoin, urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://quotes.toscrape.com/'
my_headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
TIMEOUT = 15


### 2.1. Отправляем GET-запрос

`requests.get()` возвращает объект `Response`, а не просто HTML-строку.
`timeout=15` ограничивает ожидание сетевых операций.


In [34]:
response = requests.get(
    BASE_URL,
    headers=my_headers,
    timeout=TIMEOUT,
)
print(response)
print('Код:', response.status_code)
response.raise_for_status()


<Response [200]>
Код: 200


### 2.2. Что можно узнать из ответа

| Атрибут / метод | Результат |
|---|---|
| `response.status_code` | Числовой HTTP-статус |
| `response.url` | Конечный URL после перенаправлений |
| `response.headers` | Заголовки ответа |
| `response.text` | Тело ответа как строка |
| `response.content` | Тело ответа как байты, например для картинки |
| `response.encoding` | Кодировка, используемая для декодирования `.text` |
| `response.raise_for_status()` | Исключение для статусов 4xx и 5xx |


In [35]:
print('Адрес:', response.url)
print('Тип данных:', response.headers.get('Content-Type'))
print('Кодировка:', response.encoding)
print('Перенаправления:', [r.status_code for r in response.history])
print(response.text[:500])


Адрес: https://quotes.toscrape.com/
Тип данных: text/html; charset=utf-8
Кодировка: utf-8
Перенаправления: []
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


### 2.3. Параметры запроса

Если сайт поддерживает параметры после `?`, передавайте их словарём `params`:
библиотека сама закодирует пробелы, кириллицу и специальные символы.


In [36]:
dictionary_response = requests.get(
    'https://www.multitran.com/m.exe',
        params={'ll1': '2', 'll2': 1, 's': 'crawl', 'l2': 1},
        timeout=TIMEOUT,
)
print(dictionary_response.text)

<!doctype html>
<html lang="en" xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=5.0">
<title>crawl &ndash; English translation</title>
<meta property="og:title" content="crawl &ndash; English translation"><script>
document.cookie="langs=2 1;expires=Wed, 15 Mar 2045 13:16:41 GMT;path=/"
</script>
<meta name="keywords" lang = "en" content="crawl,Multitran,dictionary,translation,Russian,English">
<meta name="description" content="Translations of ‘crawl’ in Multitran dictionary (Russian-English). Word forms, pronunciations, examples in different areas and phrases, forum discussions.">
<link rel="canonical" href="https://www.multitran.com/dictionary/russian-english/crawl"/>
<link rel="alternate" hreflang="x-default" href="https://www.multitran.com/dictionary/russian-english/crawl"/>
<link rel="alternate" hreflang="en" href="https://www.multitran.com/dictionary/russian-english/crawl"/>
<li

## 3. Повторяем BeautifulSoup

Упрощённо блок одной цитаты выглядит так:

```html
<div class="quote">
  <span class="text">“Текст цитаты”</span>
  <span><small class="author">Имя автора</small>
    <a href="/author/Author-Name">(about)</a></span>
  <div class="tags"><a class="tag">literature</a></div>
</div>
```


In [38]:
soup = BeautifulSoup(response.text, 'html.parser')
quote_blocks = soup.find_all('div', class_='quote')
print('Цитат на странице:', len(quote_blocks))
if not quote_blocks:
    raise ValueError('Цитаты не найдены: проверьте адрес и HTML ответа.')

first_quote = quote_blocks[0]
print(first_quote.prettify()[:1800])


Цитат на странице: 10
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
  <a class="tag" href="/tag/change/page/1/">
   change
  </a>
  <a class="tag" href="/tag/deep-thoughts/page/1/">
   deep-thoughts
  </a>
  <a class="tag" href="/tag/thinking/page/1/">
   thinking
  </a>
  <a class="tag" href="/tag/world/page/1/">
   world
  </a>
 </div>
</div>



In [39]:
text = first_quote.find('span', class_='text').get_text(strip=True)
author = first_quote.find('small', class_='author').get_text(strip=True)
tags = [tag.get_text(strip=True) for tag in first_quote.find_all('a', class_='tag')]

print(text)
print(author)
print(tags)


“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Albert Einstein
['change', 'deep-thoughts', 'thinking', 'world']


### 3.1. CSS-селекторы: как указать нужные элементы

**CSS** (Cascading Style Sheets) — язык описания оформления HTML-документов.
В CSS правило состоит из селектора и свойств оформления. Например,
`.quote { color: blue; }` задаёт синий цвет текста элементам с классом `quote`.
**Селектор** — выражение, которое задаёт условия выбора элементов документа.
При сборе данных нам нужно не оформление, а этот способ поиска. Один и тот же
базовый синтаксис используют BeautifulSoup и браузер, управляемый через Selenium.

Атрибут `class` объединяет элементы в группы. У элемента может быть несколько
классов: `<div class="quote featured">`. Атрибут `id` — идентификатор конкретного
элемента, который по правилам HTML должен быть уникальным в пределах документа.
Названия классов и идентификаторов задаёт разработчик сайта: сами по себе
слова `quote` или `text` не имеют специального значения для HTML.

| Селектор | Что выбирает |
|---|---|
| `span` | Все элементы с тегом `span` |
| `.quote` | Элементы с классом `quote`, независимо от тега |
| `div.quote` | Элементы `div` с классом `quote` |
| `#main` | Элемент с `id="main"` |
| `.quote.featured` | Элементы, у которых есть **оба** класса |
| `.quote .text` | Элементы с классом `text` внутри `.quote` на любой глубине |
| `.quote > .text` | Только непосредственные дочерние элементы `.quote` с классом `text` |
| `a[href]` | Ссылки, у которых есть атрибут `href` |
| `a[href^="/author/"]` | Ссылки, у которых `href` начинается с `/author/` |
| `.text, .author` | Все элементы с классом `text` **или** `author` |

Пробел важен: `.quote.featured` выбирает один элемент с двумя классами,
а `.quote .featured` — элемент с классом `featured` **внутри** элемента `.quote`.
Селекторы описывают структуру, а не внешний вид и не смысл текста.

В BeautifulSoup `select()` возвращает список совпадений (возможно, пустой),
а `select_one()` — первое совпадение или `None`. Поиск можно выполнять
как во всём документе, так и внутри уже найденной карточки.


In [40]:
css_blocks = soup.select('div.quote')
print(css_blocks[0])


<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</div>


### 3.2. Относительные ссылки

`/author/Albert-Einstein` — не полный URL. Используем `urljoin(адрес_текущей_страницы, href)`:
он учитывает относительные пути и завершающий `/`.
Склеивать адреса через `+` ненадёжно: ссылка может начинаться с `/`, `../`
или уже быть абсолютной.


In [42]:
author_link = first_quote.find('a', href=True)
author_url = urljoin(response.url, author_link['href'])
print(author_link['href'])
print(author_url)

print(urljoin('https://quotes.toscrape.com/page/2/', '/page/3/'))
print(urljoin('https://example.org/catalog/', '../about/'))


/author/Albert-Einstein
https://quotes.toscrape.com/author/Albert-Einstein
https://quotes.toscrape.com/page/3/
https://example.org/about/


### 3.3. Отдельная функция для парсинга

In [43]:
def parse_quotes(html, page_url):
    page_soup = BeautifulSoup(html, 'html.parser')
    blocks = page_soup.find_all('div', class_='quote')
    if not blocks:
        raise ValueError(f'На странице нет блоков цитат: {page_url}')

    records = []
    for block in blocks:
        text_node = block.find('span', class_='text')
        author_node = block.find('small', class_='author')
        link_node = block.find('a', href=True)
        if text_node is None or author_node is None or link_node is None:
            raise ValueError(f'Изменилась структура цитаты: {page_url}')
        records.append({
            'text': text_node.get_text(strip=True),
            'author': author_node.get_text(strip=True),
            'tags': [tag.get_text(strip=True) for tag in block.find_all('a', class_='tag')],
            'author_url': urljoin(page_url, link_node['href'].rstrip('/') + '/'),
            'source_url': page_url,
        })
    return records


first_page_records = parse_quotes(response.text, response.url)
first_page_records[0]


{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
 'author': 'Albert Einstein',
 'tags': ['change', 'deep-thoughts', 'thinking', 'world'],
 'author_url': 'https://quotes.toscrape.com/author/Albert-Einstein/',
 'source_url': 'https://quotes.toscrape.com/'}

**Мини-практика.** 

1. Создайте DataFrame из `first_page_records`.
2. Выведите уникальные имена авторов.
3. Найдите цитаты с тегом `life`.


In [26]:
# Ваш код


## 4. Что такое краулинг

**Скрейпинг (scraping)** — извлечение нужных данных со страниц.
**Парсинг (parsing)** — разбор структуры документа, например HTML.
На практике эти этапы часто объединены в одной программе.

**Краулинг (crawling)** — автоматический обход страниц по заданной стратегии.

### Стратегии обхода

1. **По номерам страниц:** `/page/1/`, `/page/2/`, … — если схема известна.
2. **По ссылке «Далее»:** извлекать её с текущей страницы. 
3. **Каталог:** собрать ссылки на страницы, а затем посетить их последовательно.

### 4.1. Правила аккуратного сбора

- Проверьте правила сайта, условия использования данных.
- Делайте паузы, чтобы не положить сервер.
- Сохраняйте адреса источников, время сбора и границы выборки.


### 4.2. Функция загрузки: пауза, проверка и кэш

Кэш — словарь `URL → HTML`: при повторном запуске в том же ядре не скачиваем
страницу заново. После перезапуска ядра этот кэш исчезнет.

Перед каждым новым запросом проверяем, что адрес остался на нужном сайте,
делаем паузу, а после запроса проверяем статус и формат. Для простоты при любой
сетевой ошибке краулер останавливается: неполный результат не должен выглядеть
полным. Автоматические перенаправления отключаем, чтобы случайно не перейти
на другой сайт.


In [52]:
session = requests.Session()
session.headers.update(my_headers)

def get_html(url):
    time.sleep(1)
    result = session.get(url, timeout=TIMEOUT, allow_redirects=False)
    return result.text


### 4.3. Находим следующую страницу

Напоминание о CSS-селекторах: `li.next a` означает «ссылка `a` внутри элемента
`li` с классом `next`». `select_one()` возвращает первый найденный элемент или `None`.
На последней странице ссылки «Далее» нет — это нормальное условие завершения.


In [47]:
def find_next_page(html, page_url):
    page_soup = BeautifulSoup(html, 'html.parser')
    next_link = page_soup.select_one('li.next a')
    if next_link is None:
        return None
    return urljoin(page_url, next_link['href'])


find_next_page(response.text, BASE_URL)


'https://quotes.toscrape.com/page/2/'

### 4.4. Собираем краулер

В `visited` храним посещённые адреса, чтобы не зациклиться.

In [53]:
def crawl_quotes(start_url):
    records = []
    current_url = start_url

    while current_url is not None:
        html = get_html(current_url)
        page_records = parse_quotes(html, current_url)
        records.extend(page_records)
        current_url = find_next_page(html, current_url)

    return records


df = pd.DataFrame(crawl_quotes(BASE_URL))
df.head()


,text,author,tags,author_url,source_url
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]",https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com/
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]",https://quotes.toscrape.com/author/J-K-Rowling/,https://quotes.toscrape.com/
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]",https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com/
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]",https://quotes.toscrape.com/author/Jane-Austen/,https://quotes.toscrape.com/
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]",https://quotes.toscrape.com/author/Marilyn-Mon...,https://quotes.toscrape.com/


In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        100 non-null    str   
 1   author      100 non-null    str   
 2   tags        100 non-null    object
 3   author_url  100 non-null    str   
 4   source_url  100 non-null    str   
dtypes: object(1), str(4)
memory usage: 25.8+ KB


## 5. Практика: кто кого качает? мы поэзию или она нас?

Возьмём электронную библиотеку [slova.org](https://slova.org.ru/harms/) и обкачаем все стихотворения Д. Хармса.


## 6. Selenium: собираем данные через браузер

### 6.1. Зачем нужен Selenium

`requests` скачивает HTML, но не выполняет JavaScript. Если страница собирается
скриптом или содержимое появляется после нажатия кнопки, нужен настоящий браузер.
**Selenium** управляет браузером из Python: открывает страницу, находит элементы,
нажимает кнопки и читает результат уже после работы скриптов. Он медленнее
`requests` и требует больше памяти, поэтому используется, только когда без него
не обойтись.


### 6.2. Установка и запуск драйвера

Установите Selenium в среду текущего ядра. Команды Python передаются браузеру
через **WebDriver** — стандарт управления браузером.

- **Chrome** — драйвер подбирается автоматически, нужен интернет при первом запуске.
- **Safari** — в macOS драйвер уже встроен, но один раз нужно разрешить управление:
  Safari → Настройки → Расширенные → «Показывать меню «Разработка» в строке меню»,
  затем в меню «Разработка» — «Разрешить удалённую автоматизацию».
  Safari работает только на macOS и всегда открывает видимое окно.

Функция `create_driver()` ниже создаёт драйвер: поменяйте `'chrome'` на `'safari'`,
чтобы работать в Safari.


In [55]:
# %pip install -U selenium

from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def create_driver(browser='chrome'):
    """Создаёт драйвер браузера: 'chrome' или 'safari'."""
    if browser.lower() == 'safari':
        return webdriver.Safari()
    return webdriver.Chrome()


def click(driver, element):
    """Обычный клик по элементу.

    В Safari некоторые элементы нажимаются только через JavaScript,
    поэтому используем запасной вариант.
    """
    try:
        element.click()
    except WebDriverException:
        driver.execute_script('arguments[0].click();', element)


### 6.3. Как работать с элементами

В практике ниже достаточно нескольких вызовов:

- `driver.get(url)` — открыть страницу;
- `driver.find_elements(By.CSS_SELECTOR, '...')` — найти все элементы по селектору;
- `element.text` — прочитать видимый текст элемента;
- `element.click()` — нажать на элемент;
- `driver.page_source` — получить текущий HTML страницы;
- `driver.quit()` — закрыть браузер.

После нажатия кнопки страница меняется, поэтому ждём нужного состояния:
`WebDriverWait(driver, 20).until(условие)`. Например,
`EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.post'))` ждёт появления
карточек, а `EC.url_changes(старый_адрес)` — смены адреса.


## 7. Обкачаем новости ВШЭ

Возьмём сайт [новостей НИУ ВШЭ](https://www.hse.ru/news/). На нём новости собраны
в список, а переход к следующей порции делается кнопкой «Следующая страница».

План: открыть страницу, собрать новости первой страницы, нажать кнопку
«Следующая страница», дождаться перехода и собрать новости второй страницы.


### 7.1. Смотрим на структуру карточки

Откройте страницу в браузере и исследуйте одну карточку новости.
Каждая новость лежит в `div.post`:

- `h2 a` — заголовок и ссылка на полный текст;
- `.post-meta__day`, `.post-meta__month`, `.post-meta__year` — дата;
- `.post__text` — краткое описание.

Кнопка перехода — `div.posts__control a.button` с текстом «Следующая страница».


In [66]:
HSE_NEWS_URL = 'https://www.hse.ru/news/'


def parse_hse_page(page_source, page_url):
    soup = BeautifulSoup(page_source, 'html.parser')
    records = []
    for post in soup.select('div.post'):
        title_node = post.select_one('h2 a')
        text_node = post.select_one('.post__text')

        date_parts = [
            node.get_text(strip=True)
            for node in post.select('.post-meta__day, .post-meta__month, .post-meta__year')
        ]
        records.append({
            'title': title_node.get_text(' ', strip=True),
            'url': urljoin(page_url, title_node['href']),
            'date': ' '.join(part for part in date_parts if part),
            'lead': text_node.get_text(' ', strip=True) if text_node else None,
            'source_url': page_url,
        })
    return records


def close_cookie_banner(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, 'span.gdpr_bar__close')
    if buttons:
        click(driver, buttons[0])
        print('Закрыли сообщение о cookies.')


### 7.2. Открываем страницу и собираем первую порцию

Создаём драйвер. По умолчанию используется Chrome; чтобы открыть страницу в Safari,
замените `create_driver('chrome')` на `create_driver('safari')`.


In [67]:
driver = create_driver('chrome')
driver.set_page_load_timeout(40)
wait = WebDriverWait(driver, 20)


In [68]:
driver.get(HSE_NEWS_URL)
close_cookie_banner(driver)

posts = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.post')))
hse_records = parse_hse_page(driver.page_source, driver.current_url)
print(f'Страница 1: {len(hse_records)} новостей — {driver.current_url}')


Закрыли сообщение о cookies.
Страница 1: 10 новостей — https://www.hse.ru/news/


### 7.3. Нажимаем кнопку «Следующая страница»

Кнопка найдена по селектору `div.posts__control a.button`. После нажатия
страница меняется, поэтому ждём, пока изменится адрес, и заново находим карточки:
старые объекты `WebElement` после перехода больше не действительны.


In [69]:
next_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.posts__control a.button'))
)
print('Нажимаем кнопку:', next_button.text.strip())

current_url = driver.current_url
click(driver, next_button)
wait.until(EC.url_changes(current_url))
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.post')))

page2_records = parse_hse_page(driver.page_source, driver.current_url)
hse_records.extend(page2_records)
print(f'Страница 2: {len(page2_records)} новостей — {driver.current_url}')


Нажимаем кнопку: Следующая страница
Страница 2: 10 новостей — https://www.hse.ru/news/page2.html


### 7.4. Складываем в таблицу и завершаем работу


In [70]:
hse_df = pd.DataFrame(hse_records)
print('Размер таблицы:', hse_df.shape)
hse_df[['date', 'title', 'source_url']].head()


Размер таблицы: (20, 5)


,date,title,source_url
0,21 окт 2026,Ученые НИУ ВШЭ систематизировали методы защиты...,https://www.hse.ru/news/
1,21 окт 2026,НИУ ВШЭ обучит до 100 участников инициативы «С...,https://www.hse.ru/news/
2,20 окт 2026,«Наши работы можно назвать исследованиями на п...,https://www.hse.ru/news/
3,20 окт 2026,За гранью возможного: в Вышке прошел финал IX ...,https://www.hse.ru/news/
4,20 окт 2026,Студенты четырех ведущих вузов представят Росс...,https://www.hse.ru/news/


In [ ]:
# Завершаем браузерный сеанс. 
driver.quit()
